# Quantum Random Number Generator on IBM Quantum
### The entropy source for the Quantum-Safe Channel + Storage bundle

**Environment:** qBraid · IBM Quantum (Heron preferred)
**Time budget:** ~1 minute of QPU time from the 10-minute monthly free budget
**Result:** ~100,000 hardware-measured random bits, debiased and statistically validated, output as ready-to-paste hex keys for the BB84 and PQC notebooks.

## Why this notebook

The other two notebooks in the bundle both depend on cryptographically strong entropy:

- The **BB84 notebook** uses random bits for Alice's basis and bit choices (Z/X, 0/1) per round.
- The **PQC notebook** (§13.8 of the operator guide) flags that its demo-mode `K_QKD = os.urandom(32)` falls back to the operating system's CSPRNG, which is a known production limitation.

This notebook closes that gap. Instead of relying on a classical PRNG seeded by ambient noise, it generates entropy directly from a quantum measurement: a single qubit prepared in $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ has exactly $\frac{1}{2}$ probability of yielding either outcome when measured in the computational basis — and that probability is, under the standard interpretation of quantum mechanics, *fundamentally* random rather than merely *unpredictably deterministic*.

The output is formatted to drop directly into both companion notebooks: a 256-bit hex string for the BB84 sifted-key input (Path B of §2) and additional 256-bit keys for the PQC `K_QKD` input (Path B of §2 there).

## 1. Imports

In [ ]:
# Uncomment if needed
%pip install -q qiskit qiskit-ibm-runtime matplotlib numpy scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from binascii import hexlify
from scipy import stats

from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

print("Imports OK")

## 2. Connect & check usage *before*

In [ ]:
service = QiskitRuntimeService()
print("Connected. Instance:", service.active_account().get("instance", "(default)"))

In [ ]:
def used_seconds_this_month(svc):
    """Sum reported QPU usage_seconds across this calendar month's jobs."""
    now = datetime.now(timezone.utc)
    start = datetime(now.year, now.month, 1, tzinfo=timezone.utc)
    total = 0.0
    try:
        jobs = svc.jobs(created_after=start, limit=200)
    except Exception as e:
        print(f"  (could not enumerate jobs: {e})")
        return None
    for j in jobs:
        try:
            u = j.usage()
            if isinstance(u, dict):
                total += float(u.get("seconds", u.get("quantum_seconds", 0)) or 0)
            else:
                total += float(u or 0)
        except Exception:
            pass
    return total

used_before = used_seconds_this_month(service)
if used_before is not None:
    print(f"Used this month:        {used_before:7.1f} s   ({used_before/60:5.2f} min)")
    print(f"Remaining (10 min plan): {max(0, 600 - used_before):7.1f} s   ({max(0, 600 - used_before)/60:5.2f} min)")

## 3. Backend and qubit selection

QRNG quality is dominated by the readout/measurement error of the chosen qubit — a noisy measurement biases the raw distribution and reduces the entropy yield after debiasing. We pick the qubit with the lowest reported readout error.

In [ ]:
PREFERRED = ["ibm_fez", "ibm_marrakesh", "ibm_torino"]

backend = None
for name in PREFERRED:
    try:
        b = service.backend(name)
        if b.status().operational:
            backend = b
            break
    except Exception:
        continue
if backend is None:
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=1)

print(f"Backend       : {backend.name}")
print(f"Pending jobs  : {backend.status().pending_jobs}")

In [ ]:
def best_single_qubit(backend):
    target = backend.target
    best, best_err = None, float("inf")
    if "measure" in target.operation_names:
        for q, props in target["measure"].items():
            if props is None or props.error is None:
                continue
            if props.error < best_err:
                best_err, best = props.error, q[0]
    if best is None:
        return 0, None
    return best, best_err

PHYS_QUBIT, meas_err = best_single_qubit(backend)
print(f"Selected physical qubit: {PHYS_QUBIT}   (measurement error: "
      + (f"{meas_err:.2e}" if meas_err else "n/a") + ")")

## 4. The QRNG circuit

Trivially simple:

1. Initialize qubit in $|0\rangle$ (the default).
2. Apply Hadamard to put it in $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$.
3. Measure in the computational basis.

Each shot yields one bit. With $N$ shots we get $N$ raw bits.

In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)
print(qc.draw())

## 5. Transpile + submit  ⏱️ *metered step*

Default `SHOTS = 100_000` — the maximum per job on the Open Plan. A single-shot QRNG measurement is one of the fastest workloads IBM Quantum runs, so this typically meters at ~30–90 seconds of QPU time.

To consume more or less time, change SHOTS:
- `SHOTS = 50_000` → ~15–45 s
- `SHOTS = 100_000` → ~30–90 s *(default — targets your 1-minute budget)*
- Re-run the submit cell for additional sets (a second job adds another ~30–90 s)

In [ ]:
SHOTS = 150_000

pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=2,
    initial_layout=[PHYS_QUBIT],
)
isa_qc = pm.run(qc)

sampler = Sampler(mode=backend)
sampler.options.default_shots = SHOTS
# Turn OFF dynamical decoupling — irrelevant for a depth-1 circuit and adds no value here
sampler.options.dynamical_decoupling.enable = False
# Turn OFF twirling — we *want* the raw measurement statistics, not a symmetrized version
sampler.options.twirling.enable_gates = False
sampler.options.twirling.enable_measure = False

job = sampler.run([isa_qc])
print(f"Submitted job: {job.job_id()}")
print(f"  backend = {backend.name}")
print(f"  qubit   = {PHYS_QUBIT}")
print(f"  shots   = {SHOTS:,}")
print(f"  status  = {job.status()}")

In [ ]:
# Block until done (QPU time only counted while executing, not while queued)
result = job.result()
print("Done. Status:", job.status())

## 6. Extract raw bits

In [ ]:
# SamplerV2 returns per-shot bit data; convert to a numpy array of 0/1
data = result[0].data.c  # 'c' is the default classical register name from .measure(0,0)
bitarray = data.array.flatten()  # shape (SHOTS,) with values 0 or 1
raw_bits = np.asarray(bitarray, dtype=np.uint8)
N = len(raw_bits)
print(f"Collected {N:,} raw bits")
print(f"First 80 bits: {''.join(str(b) for b in raw_bits[:80])}")

## 7. Bias analysis

A perfect $|+\rangle$ measured in Z gives $P(0) = P(1) = \frac{1}{2}$ exactly. Real qubits show small asymmetry — typically a few tenths of a percent — driven mostly by measurement infidelity (the chip's readout discriminator skewing toward one outcome) and to a lesser extent gate calibration.

In [ ]:
n_zeros = int((raw_bits == 0).sum())
n_ones  = int((raw_bits == 1).sum())
p_zero  = n_zeros / N
p_one   = n_ones / N
bias    = p_one - 0.5
# Approximate standard error for a Bernoulli sample
se      = np.sqrt(0.25 / N)
z_score = bias / se

print(f"  N(0)    = {n_zeros:>7,}   ({p_zero*100:6.3f}%)")
print(f"  N(1)    = {n_ones:>7,}   ({p_one*100:6.3f}%)")
print(f"  bias    = P(1) - 0.5 = {bias:+.5f}  ({bias*100:+.4f} pp)")
print(f"  z-score = {z_score:+.2f}   (>{2.58 if abs(z_score)>0 else 0:.2f}σ would be 'significant' at p<0.01)")

## 8. Von Neumann debiasing

Standard technique (von Neumann, 1951) to remove any consistent bias from independent bits regardless of $P(0)$:

- pair up bits: $(b_{2i}, b_{2i+1})$ for $i = 0, 1, 2, \ldots$
- if $(0, 1)$ → output 0
- if $(1, 0)$ → output 1
- if $(0, 0)$ or $(1, 1)$ → discard

For an unbiased source we expect to keep $\frac{1}{2} \cdot \frac{1}{2} \cdot 2 = \frac{1}{2}$ of the pairs (one quarter from each cross-type), i.e. $\frac{N}{4}$ output bits from $N$ input bits. For a biased source we keep proportionally fewer.

In [ ]:
def von_neumann_debias(bits: np.ndarray) -> np.ndarray:
    """Von Neumann's classical extractor: (0,1)→0, (1,0)→1, (0,0)/(1,1)→discard."""
    if len(bits) % 2:
        bits = bits[:-1]
    pairs = bits.reshape(-1, 2)
    kept = pairs[:, 0] != pairs[:, 1]      # keep cross-pairs only
    return pairs[kept, 0].astype(np.uint8)  # in a (a,b) pair with a≠b, output a (so (0,1)→0, (1,0)→1)

clean_bits = von_neumann_debias(raw_bits)
M = len(clean_bits)
yield_pct = 100 * M / N

print(f"  raw bits     : {N:>7,}")
print(f"  debiased bits: {M:>7,}   (yield: {yield_pct:.2f}%, theoretical max 50%)")
print(f"  P(0) after   : {(clean_bits == 0).mean()*100:.3f}%")
print(f"  P(1) after   : {(clean_bits == 1).mean()*100:.3f}%")

## 9. Statistical tests

A minimum-honesty battery on the debiased output. None of these *prove* randomness — they only flag the obvious failure modes. NIST SP 800-22 specifies a much fuller suite for production qualification.

1. **Monobit (frequency) test** — does $P(0) \approx P(1) \approx \frac{1}{2}$?
2. **Chi-squared on byte frequencies** — do all 256 byte values appear roughly equally often?
3. **Runs test** — are run lengths consistent with i.i.d. coin-flipping?
4. **Lag-1 autocorrelation** — is each bit independent of the previous one?

In [ ]:
def monobit_test(bits):
    s = bits.sum() - len(bits)/2
    z = s / np.sqrt(len(bits)/4)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p

def byte_chi2(bits):
    # Pack into bytes; drop trailing partial byte
    n_bytes = len(bits) // 8
    b = bits[:n_bytes*8].reshape(n_bytes, 8)
    byte_vals = (b * (1 << np.arange(7, -1, -1))).sum(axis=1)
    observed, _ = np.histogram(byte_vals, bins=np.arange(257))
    expected = np.full(256, n_bytes / 256)
    chi2 = ((observed - expected) ** 2 / expected).sum()
    p = 1 - stats.chi2.cdf(chi2, df=255)
    return chi2, p, n_bytes

def runs_test(bits):
    # Number of runs (transitions + 1) vs expected for fair iid
    n1 = bits.sum()
    n0 = len(bits) - n1
    transitions = (bits[1:] != bits[:-1]).sum()
    runs = transitions + 1
    n = len(bits)
    expected_runs = 2 * n0 * n1 / n + 1
    var_runs = (2 * n0 * n1 * (2 * n0 * n1 - n)) / (n**2 * (n - 1))
    z = (runs - expected_runs) / np.sqrt(var_runs)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return runs, expected_runs, z, p

def autocorr_lag1(bits):
    x = bits.astype(np.int32) * 2 - 1  # map 0→-1, 1→+1
    r = np.corrcoef(x[:-1], x[1:])[0, 1]
    z = r * np.sqrt(len(bits) - 1)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return r, z, p

# Run all four on the debiased bits
print("=" * 64)
print("STATISTICAL TESTS — debiased bits")
print("=" * 64)

z, p = monobit_test(clean_bits)
print(f"\n[1] Monobit (frequency)")
print(f"     z-score  = {z:+.3f}")
print(f"     p-value  = {p:.4f}   ({'PASS' if p > 0.01 else 'FAIL'} at α=0.01)")

chi2, p, nb = byte_chi2(clean_bits)
print(f"\n[2] Byte-frequency chi-squared  ({nb:,} bytes, 255 df)")
print(f"     chi²     = {chi2:.2f}   (expected ≈ 255)")
print(f"     p-value  = {p:.4f}   ({'PASS' if p > 0.01 else 'FAIL'} at α=0.01)")

runs, exp_runs, z, p = runs_test(clean_bits)
print(f"\n[3] Runs test")
print(f"     observed runs = {runs:,}")
print(f"     expected runs = {exp_runs:,.0f}")
print(f"     z-score = {z:+.3f}")
print(f"     p-value = {p:.4f}   ({'PASS' if p > 0.01 else 'FAIL'} at α=0.01)")

r, z, p = autocorr_lag1(clean_bits)
print(f"\n[4] Lag-1 autocorrelation")
print(f"     r        = {r:+.5f}   (expected 0)")
print(f"     z-score  = {z:+.3f}")
print(f"     p-value  = {p:.4f}   ({'PASS' if p > 0.01 else 'FAIL'} at α=0.01)")

## 10. Visualize

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

# Panel 1: raw bias bars
axes[0].bar(["P(0)", "P(1)"],
            [n_zeros/N*100, n_ones/N*100],
            color=["#1f77b4", "#d62728"], edgecolor="black")
axes[0].axhline(50, color="black", ls="--", lw=1)
axes[0].set_ylabel("Probability (%)")
axes[0].set_title(f"Raw bits ({N:,})\nbias = {bias*100:+.3f} pp")
axes[0].set_ylim(45, 55)
for i, v in enumerate([n_zeros/N*100, n_ones/N*100]):
    axes[0].text(i, v + 0.15, f"{v:.3f}%", ha="center", fontweight="bold", fontsize=9)

# Panel 2: debiased
p0_clean = (clean_bits == 0).mean() * 100
p1_clean = (clean_bits == 1).mean() * 100
axes[1].bar(["P(0)", "P(1)"],
            [p0_clean, p1_clean],
            color=["#2ca02c", "#2ca02c"], edgecolor="black")
axes[1].axhline(50, color="black", ls="--", lw=1)
axes[1].set_ylabel("Probability (%)")
axes[1].set_title(f"Debiased ({M:,})\nyield = {yield_pct:.1f}%")
axes[1].set_ylim(45, 55)
for i, v in enumerate([p0_clean, p1_clean]):
    axes[1].text(i, v + 0.15, f"{v:.3f}%", ha="center", fontweight="bold", fontsize=9)

# Panel 3: byte histogram (debiased)
n_bytes = M // 8
b = clean_bits[:n_bytes*8].reshape(n_bytes, 8)
byte_vals = (b * (1 << np.arange(7, -1, -1))).sum(axis=1)
axes[2].hist(byte_vals, bins=64, color="#087E8B", edgecolor="black", alpha=0.85)
axes[2].axhline(n_bytes/64, color="black", ls="--", lw=1, label="expected (uniform)")
axes[2].set_xlabel("Byte value (0–255)")
axes[2].set_ylabel("Frequency")
axes[2].set_title(f"Byte distribution\n({n_bytes:,} bytes)")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 11. Output: keys ready for the companion notebooks

Format the debiased bits into 256-bit (64-hex-character) keys you can paste directly into:

- **BB84 notebook** §2 Path B — replace `K_QKD_hex` with the first key.
- **PQC notebook** §2 Path B — replace `K_QKD_hex` similarly.

Each 256-bit key consumes 32 bytes = 256 debiased bits. With ~25k debiased bits typical from a 100k-shot run, you'll get ~95 independent keys per run.

In [ ]:
def bits_to_hex(bits: np.ndarray) -> str:
    # Pad to a multiple of 8 if needed (truncate is safer for cryptographic keys)
    n_full_bytes = len(bits) // 8
    b = bits[:n_full_bytes*8].reshape(n_full_bytes, 8)
    byte_vals = (b * (1 << np.arange(7, -1, -1))).sum(axis=1).astype(np.uint8)
    return byte_vals.tobytes().hex()

# Build keys of 256 bits each
KEY_BITS = 256
n_keys = M // KEY_BITS
print(f"Producing {n_keys} independent 256-bit keys from {M:,} debiased bits.")
print()

keys = []
for i in range(n_keys):
    chunk = clean_bits[i*KEY_BITS : (i+1)*KEY_BITS]
    keys.append(bits_to_hex(chunk))

print("=" * 70)
print("PRIMARY K_QKD (paste into BB84 §2 or PQC §2, Path B)")
print("=" * 70)
print()
print(f'K_QKD_hex = "{keys[0]}"')
print()
if n_keys > 1:
    print("Additional 256-bit keys available:")
    for i, k in enumerate(keys[1:min(6, n_keys)], start=1):
        print(f"  [{i}] {k}")
    if n_keys > 6:
        print(f"  ... and {n_keys - 6} more in `keys` variable")
print()
print(f"Total: {n_keys} independent keys, each 256 bits.")

## 12. Check QPU time consumed

In [ ]:
try:
    u = job.usage()
    if isinstance(u, dict):
        secs = float(u.get("seconds", u.get("quantum_seconds", 0)) or 0)
        print(f"This job used: {secs:6.2f} s   ({secs/60:5.3f} min)   [{u}]")
    else:
        print(f"This job used: {float(u):6.2f} s   ({float(u)/60:5.3f} min)")
except Exception as e:
    print(f"Could not read job.usage(): {e}")

used_after = used_seconds_this_month(service)
if used_before is not None and used_after is not None:
    print()
    print(f"Month total before : {used_before:7.1f} s   ({used_before/60:5.2f} min)")
    print(f"Month total after  : {used_after:7.1f} s   ({used_after/60:5.2f} min)")
    print(f"Delta              : {used_after - used_before:+7.1f} s")

## 13. What this is, and isn't

**What it is.** A *trusted-device* QRNG: the entropy comes from genuine quantum measurement on real hardware, and the post-processing (von Neumann debiasing + statistical tests) is honest enough to flag obvious failure modes. The output is suitable as a research-quality entropy source for the companion notebooks, demonstrably better than `os.urandom` for the purpose of demonstrating the bundle end-to-end.

**What it isn't.** A *device-independent* QRNG. Trusting these bits requires trusting that IBM's hardware is in fact preparing $|+\rangle$ and measuring in Z — not, say, measuring a pre-computed deterministic state. Device-independent randomness certification requires a Bell-inequality violation: the CHSH violation observed in the bundle's BB84 sibling notebook is exactly the kind of test that, when continuously monitored, certifies the *un*predictability of the measurement outcomes regardless of what's happening inside the device. See:

- Colbeck, R. (2009). *Quantum and relativistic protocols for secure multi-party computation.* DPhil thesis, U. Cambridge — first proposal of DI-QRNG via Bell violations.
- Pironio, S. et al. (2010). *Random numbers certified by Bell's theorem.* Nature 464, 1021–1024.
- Liu, Y. et al. (2018). *Device-independent quantum random number generation.* Nature 562, 548–551 — first end-to-end DI-QRNG demonstration.

**Production replacement.** Commercial QRNGs (ID Quantique Quantis, qStream, PicoQuant) are the appropriate choice for fielded systems. Their value over this notebook is mainly:

- Continuous output (not batch).
- Hardware health monitoring (live entropy estimation, tamper detection, fail-safe shutoff).
- Certified NIST SP 800-90B and BSI AIS 31 compliance.

**How this completes the bundle.** The three notebooks now form a closed loop:

1. **QRNG** (this notebook) — generates entropy on IBM Quantum hardware.
2. **BB84 QKD** — uses the entropy to establish a shared key with eavesdropper detection.
3. **Hybrid PQC wrapping** — combines the shared key with ML-KEM to wrap data against both classical and harvest-now-decrypt-later attacks.

Each notebook's output can flow into the next. For a board or regulator briefing, this is the difference between three disconnected demos and one coherent post-quantum security narrative.